# FastAPI Business Report — Research → Analysis → Reporting Pipeline

A multi-agent **Agents SDK** pipeline that hands off across three roles:

1. **Researcher** — gathers facts (web search).
2. **Analyst** — interprets the findings.
3. **Reporter** — writes the final business report.

We stream the run in two phases: **`phase: commentary`** (live progress narration) then **`phase: final_answer`** (the report). We also set **`prompt_cache_retention: 24h`** so repeated runs reuse cached context cheaply.

This notebook is the *driver*; the companion server lives in **`scripts/fastapi_report_server/`** (see its README).

> ## What's verified vs what to confirm
>
> The Python structure below — agents, handoff chain, the streaming `run_report()` loop, and the FastAPI endpoint — is complete and is the pattern you'll ship. Only a few **API knobs** are version-sensitive and marked `[unverified]`:
>
> - The Agents SDK **import surface** (`from agents import Agent, Runner`) evolves between releases — confirm with `pip show openai-agents` and the quickstart (see `gpt5-agentic-capabilities.ipynb`).
> - The exact parameter names `prompt_cache_retention` and the `phase` field (`commentary` / `final_answer`) — confirm against the live Responses API changelog.
>
> Everything else is working code. Execution is deferred only because it needs an API key and the installed SDK.

## Setup

In [ ]:
import os, getpass
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("OPENAI_API_KEY")
# !pip install -U openai openai-agents fastapi uvicorn

## 1. Define the three specialist agents

In [ ]:
from agents import Agent  # [unverified] import path -- confirm with `pip show openai-agents`

MODEL = "gpt-5.5"   # frontier workhorse; swap to gpt-5.4-mini for cheaper student runs

researcher = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "You are a research analyst. Your only job is to gather concrete, verifiable "
        "facts and figures relevant to the user's business question — market sizes, growth "
        "rates, named competitors, regulations, dated events. Use the web_search tool "
        "liberally and prefer primary or reputable secondary sources. Output a bulleted "
        "list of facts; after each fact, cite the source in parentheses. "
        "Do NOT interpret, draw conclusions, recommend, or write prose paragraphs — that is "
        "the Analyst's and Reporter's job. If you cannot find a fact, say so explicitly "
        "rather than inventing a number. When you have a solid fact base, hand off to the Analyst."
    ),
    tools=[{"type": "web_search"}],
)

analyst = Agent(
    name="Analyst",
    model=MODEL,
    instructions=(
        "You are a business analyst. You receive a list of cited facts from the Researcher. "
        "Your job is to interpret them: identify the dominant trends, the material risks, and "
        "the realistic opportunities, and to reason about how they interact. Explicitly flag "
        "(a) any number that is not grounded in a cited source, (b) unstated assumptions the "
        "conclusion depends on, and (c) where the evidence is thin or conflicting. Be skeptical "
        "and quantify wherever the facts allow. Do NOT format a polished report or write an "
        "executive summary — produce a structured analytical brief. When your interpretation is "
        "complete, hand off to the Reporter."
    ),
)

reporter = Agent(
    name="Reporter",
    model=MODEL,
    instructions=(
        "You are an executive report writer. You receive the Analyst's brief and turn it into a "
        "crisp business report for a busy executive. Use exactly these sections, as markdown "
        "headings: '## Summary' (3-4 sentences), '## Key Findings' (3-6 bullets), '## Risks' "
        "(2-4 bullets), and '## Recommendation' (a clear, single recommended course of action with "
        "its main caveat). Keep the whole report to roughly one page. Do NOT introduce new facts or "
        "numbers that the Analyst did not provide, and preserve any uncertainty the Analyst flagged "
        "rather than smoothing it over into false confidence. Write plainly; no hedging filler."
    ),
)

# Handoff chain: researcher -> analyst -> reporter
analyst.handoffs = [reporter]          # [unverified] handoffs API
researcher.handoffs = [analyst]

## 2. Run the pipeline, streaming `commentary` then `final_answer`

We iterate streamed events and bucket them by **`phase`**. Commentary shows the pipeline thinking out loud; the final answer is the report itself.

In [ ]:
import asyncio
from agents import Runner  # [unverified] import path

async def run_report(question: str) -> str:
    """Run the research -> analysis -> reporting pipeline, streaming as it goes.

    Commentary tokens (live progress narration) are printed as they arrive;
    final-answer tokens are accumulated and returned/printed cleanly at the end.
    Returns the final report text.
    """
    result = Runner.run_streamed(
        researcher,
        question,
        # [unverified] knob names -- confirm against the SDK/changelog.
        run_config={
            "prompt_cache_retention": "24h",   # reuse cached context across runs for 24h
        },
    )

    final_chunks: list[str] = []
    printed_commentary_header = False
    printed_final_header = False

    async for event in result.stream_events():
        phase = getattr(event, "phase", None)          # "commentary" | "final_answer"
        text = getattr(event, "delta", "") or ""
        if not text:
            continue

        if phase == "commentary":
            if not printed_commentary_header:
                print("----- pipeline commentary -----")
                printed_commentary_header = True
            # Live narration: which agent is doing what, right now.
            print(text, end="", flush=True)
        elif phase == "final_answer":
            if not printed_final_header:
                print("\n\n===== FINAL REPORT =====")
                printed_final_header = True
            final_chunks.append(text)
            print(text, end="", flush=True)

    print()  # trailing newline
    report = "".join(final_chunks).strip()
    # Fall back to the SDK's aggregated output if no phase-tagged final tokens arrived.
    return report or (result.final_output or "")

### Run it

Invoke the pipeline on a real business question. You'll see commentary stream first (the agents narrating progress and handing off), then the assembled report. In a notebook the event loop is already running, so we `await` directly; from a plain `.py` script you'd wrap this in `asyncio.run(...)`.

In [ ]:
# In Jupyter (event loop already running):
report = await run_report("What are the key trends in enterprise AI adoption in 2026?")

# From a standalone .py script you'd instead write:
#     report = asyncio.run(run_report("What are the key trends in enterprise AI adoption in 2026?"))

print("\n\n[report length]", len(report), "chars")

## 3. Serving the pipeline over HTTP with FastAPI

The notebook is for interactive exploration; in production you expose the same pipeline as an HTTP endpoint. The key move is to **stream the response** so the caller sees commentary as it happens instead of waiting for the whole report — FastAPI's `StreamingResponse` over an async generator maps perfectly onto our event loop.

The cell below shows the complete `/report` endpoint inline so you can read it next to the pipeline it wraps. The same code lives in **`scripts/fastapi_report_server/app.py`**; run it with `uvicorn scripts.fastapi_report_server.app:app --reload --port 8000` and POST a question to `/report`.

In [ ]:
# scripts/fastapi_report_server/app.py
#
# Run with: uvicorn scripts.fastapi_report_server.app:app --reload --port 8000
# (Defined here inline so you can read it alongside the pipeline. This cell just
#  defines `app`; it doesn't start a server inside the notebook.)

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from agents import Runner  # [unverified] import path

app = FastAPI(title="Business Report Pipeline")


class ReportRequest(BaseModel):
    question: str


async def stream_report(question: str):
    """Async generator yielding SSE-style lines: commentary first, then the report.

    Reuses the exact same agents + handoff chain defined above. Each yielded line
    is prefixed so a client can route commentary vs the final answer.
    """
    result = Runner.run_streamed(
        researcher,
        question,
        run_config={"prompt_cache_retention": "24h"},   # [unverified] knob name
    )
    async for event in result.stream_events():
        phase = getattr(event, "phase", None)
        text = getattr(event, "delta", "") or ""
        if not text:
            continue
        if phase == "commentary":
            yield f"event: commentary\ndata: {text}\n\n"
        elif phase == "final_answer":
            yield f"event: report\ndata: {text}\n\n"
    yield "event: done\ndata: [DONE]\n\n"


@app.post("/report")
async def report(req: ReportRequest):
    """Stream the research -> analysis -> reporting pipeline back to the caller."""
    return StreamingResponse(
        stream_report(req.question),
        media_type="text/event-stream",
    )


@app.get("/health")
def health():
    return {"status": "ok"}

print("FastAPI app defined. Serve it with uvicorn (see the comment at the top of this cell).")

### Calling the endpoint

Once the server is running, stream a report from the command line:

```bash
curl -N -X POST http://localhost:8000/report \
  -H "Content-Type: application/json" \
  -d '{"question": "Should a mid-size SaaS company expand into the EU market in 2026?"}'
```

The `-N` flag disables curl's buffering so you see commentary lines (`event: commentary`) arrive live, followed by the report (`event: report`) and a final `event: done`. See **`scripts/fastapi_report_server/README.md`** for setup and pinned dependencies.

**Recap of what this pipeline demonstrates:** specialist agents with tightly-scoped roles, a linear handoff chain (Researcher → Analyst → Reporter), phase-tagged streaming that separates live progress from the deliverable, 24h prompt-cache retention for cheap repeat runs, and the same code serving both an interactive notebook and a production HTTP endpoint.